SDS210-Project: An analysis of the "Züri Wie Neu" data

In [1]:
import pandas as pd
import geopandas as gpd

#read data (all datasets which are necessary to answer questions 1-4) 
meldungen_csv = gpd.read_file("data/raw/stzh.zwn_meldungen_p.json") #name a bit misleading, but used in the code too many times to change now

quartiere_json = gpd.read_file("data/raw/stzh.adm_statistische_quartiere_v.json")
quartiere_csv = gpd.read_file("data/raw/stzh.adm_statistische_quartiere_v.csv")

fläche_csv = pd.read_csv("data/raw/bevölkerung_zh2.csv")

Quick overview on the datasets (can be skipped or commented, but important to see what we are working with)

In [2]:
#overview of columns of different datasets (can be skipped or commented, but )
print("Meldungen CSV:", meldungen_csv.columns)
print("Quartiere JSON:", quartiere_json.columns)
print("Quartiere CSV", quartiere_csv.columns)
print("Fläche CSV:", fläche_csv.columns)

Meldungen CSV: Index(['objectid', 'service_request_id', 'requested_datetime',
       'agency_sent_datetime', 'updated_datetime', 'e', 'n', 'service_code',
       'service_name', 'status', 'userid', 'title', 'detail', 'media_url',
       'interface_used', 'service_notice', 'description', 'url', 'geometry'],
      dtype='str')
Quartiere JSON: Index(['objid', 'objectid', 'geometry'], dtype='str')
Quartiere CSV Index(['objid', 'objectid', 'qname', 'qnr', 'kname', 'knr', 'geometry'], dtype='str')
Fläche CSV: Index(['RaumKategorie', 'RaumSort', 'RaumLang', 'StichtagDatJahr', 'FlaecheT',
       'FlaecheL', 'FlaecheS'],
      dtype='str')


Question 1a: Wich is the category with the highest number of reports?  

In [ ]:
#extend meldungen_csv with a 0-column 
meldungen_csv["anzahl_meldungen"] = 0


#overwrite 0-column with the number of reports 
for kategorie in meldungen_csv["service_code"].unique():  
    anzahl_meldungen = meldungen_csv[meldungen_csv["service_code"] == kategorie].shape[0] 
    meldungen_csv.loc[ #use loc for integer-based selection based on position/location
        meldungen_csv["service_code"] == kategorie, 
        "anzahl_meldungen"] = anzahl_meldungen
    
    print(f"Kategorie: {kategorie}, Anzahl Meldungen: {anzahl_meldungen}")


Kategorie: Strasse/Trottoir/Platz, Anzahl Meldungen: 9870
Kategorie: Abfall/Sammelstelle, Anzahl Meldungen: 27339
Kategorie: Grünflächen/Spielplätze, Anzahl Meldungen: 7238
Kategorie: Beleuchtung/Uhren, Anzahl Meldungen: 5407
Kategorie: Graffiti, Anzahl Meldungen: 3759
Kategorie: Signalisation/Lichtsignal, Anzahl Meldungen: 10975
Kategorie: Brunnen/Hydranten, Anzahl Meldungen: 1289
Kategorie: VBZ/ÖV, Anzahl Meldungen: 1882
Kategorie: Allgemein, Anzahl Meldungen: 3969
Kategorie: Schädlinge, Anzahl Meldungen: 895


Question 1b: Which are the top-3 'Quartiere'/'Kreise' with the most reports? 

IMPROVEMENTS: listen sortieren nach anzahl meldungen, nur top 3 anzeigen

In [ ]:
from shapely.geometry import Point
import geopandas as gpd

#ensure the correct CRS
quartiere_ch = quartiere_json.to_crs(epsg = 2056)


#merge quartiere_json and _csv to combine spatial data and important attributes (+ set active geometry)
quartiere_gdf = quartiere_json.merge(
    quartiere_csv, on = "objid", how = "left").set_geometry("geometry_x").to_crs(epsg = 2056) 


#create geometries/a geodatagrame from meldungen_csv
meldungen_gdf = gpd.GeoDataFrame(
    meldungen_csv,
    geometry=gpd.points_from_xy(meldungen_csv["e"], meldungen_csv["n"]),
    crs=quartiere_ch.crs)
meldungen_ch = meldungen_gdf.to_crs(epsg = 2056)


#spatial join of meldungen und quartiere
meldungen_quartier_join = gpd.sjoin(
    meldungen_ch, #left
    quartiere_gdf, #right
    how = "inner", predicate = "intersects")


#calculate number of reports (meldungen) per 'Quartier' and 'Kreis' 
meldungen_pro_quartier = (meldungen_quartier_join.groupby("qname").size().reset_index(name = "anzahl_meldungen_quartier"))
display(meldungen_pro_quartier)

meldungen_pro_kreis = (meldungen_quartier_join.groupby("kname").size().reset_index(name = "anzahl_meldungen_kreis"))
display(meldungen_pro_kreis)


#add the newly calculated values to the join 
join_updated = meldungen_quartier_join.merge(
    meldungen_pro_kreis,
    on = "kname",
    how = "left")

join_updated = join_updated.merge(
    meldungen_pro_quartier,
    on = "qname",
    how = "left")


#reduce the number of columns of meldungen_quartier_join (only keep the necessary ones)
meldungen_quartier_join = join_updated[["objectid", "requested_datetime", "e", "n", "service_code", "geometry", "anzahl_meldungen", "index_right", "qname", "qnr", "kname", "knr", "geometry_y", "anzahl_meldungen_quartier", "anzahl_meldungen_kreis"]]


,qname,anzahl_meldungen_quartier
0,Affoltern,2419
1,Albisrieden,2048
2,Alt-Wiedikon,2502
3,Altstetten,4094
4,City,1741
5,Enge,2756
6,Escher Wyss,1562
7,Fluntern,1254
8,Friesenberg,1415
9,Gewerbeschule,2230


,kname,anzahl_meldungen_kreis
0,Kreis 1,5738
1,Kreis 10,6394
2,Kreis 11,8026
3,Kreis 12,2764
4,Kreis 2,6225
5,Kreis 3,9152
6,Kreis 4,10569
7,Kreis 5,3792
8,Kreis 6,5110
9,Kreis 7,5714


Question 2: Is the density of reports higher around the lake than elsewhere? 
Assumption: "Around the lake" includes 'Kreise' 1,2 and 8

In [14]:
#create subsets to reduce the size of the datasets 
quartiere_subset = quartiere_gdf[["objid", "geometry_x", "kname", "knr"]]

fläche_subset = fläche_csv[fläche_csv["RaumLang"].str.contains("Kreis", na = False)]


#join the fläche_subset with the meldungen_quartier_join
join_fläche_meldungen = fläche_subset.merge(
    meldungen_quartier_join[["kname", "anzahl_meldungen_kreis"]],
    left_on = "RaumLang", 
    right_on = "kname", 
    how = "left")


#calculate the report density per area (here:hectares)
join_fläche_meldungen["meldungsdichte_kreis"] = join_fläche_meldungen["anzahl_meldungen_kreis"]/join_fläche_meldungen["FlaecheT"]


#merge the report density with the meldungen_quartier_join 
quartiere_subset = quartiere_subset.merge(
    join_fläche_meldungen[["meldungsdichte_kreis", "RaumLang"]],
    left_on = "kname", 
    right_on = "RaumLang",
    how = "inner")


visualization to question 2 

PROBLEM: GEIT NED, STÜRZT AB SOBAUD IS WETT PLOTTE

In [ ]:
#create the empty subplots 
fig, ax = plt.subplots(figsize=(10, 8)) #set up figure and axes


#confidence intervals 
dmin = quartiere_subset["meldungsdichte_kreis"].quantile(0.02)
dmax = quartiere_subset["meldungsdichte_kreis"].quantile(0.98)


#create a legend 
legend_options = {
    "label": "Report Density per Stadtkreis Normalized per Population",
    "orientation": "horizontal",
    "shrink": 0.6,
    "pad" : 0.05}


#combine the previous elements to a plot
charte_plot = quartiere_subset.plot(
    ax = ax, 
    column = "meldungsdichte_kreis",
    vmin = dmin,
    vmax = dmax,
    cmap = "viridis",
    legend = True,
    legend_kwds = legend_options,
    edgecolor = "grey",
    linewidth = 0.1)

Question 3: How does the report density differ between august (summer) and january (winter)?

QUESTION: I HA D FUNKTION UND MACHES MIT ERE LÄÄRE LISTE. I WETT ABER KE LISTE SONDERN ES DATAFRAME. CHANI S PRINZIP ÜBERNÄÄ UND WENN JO, WIE MACHI S GLICHE EIFACH MIT EMNE DATAFRAME (STATT LISTE)

In [ ]:
import geopandas as gpd
import pandas as pd

# parse date in meldungen_quartier_join
meldungen_quartier_join["requested_datetime"] = pd.to_datetime(
    meldungen_quartier_join["requested_datetime"], format = "%Y%m%d%H%M%S")


#create a subset which only contains the data about the 'Kreise' around the lake (1,2,8)
kreise_see = meldungen_quartier_join[meldungen_quartier_join["knr"].isin(["1", "2", "8"])]


#August und Januar (manually)
august = meldungen_quartier_join[(meldungen_quartier_join["requested_datetime"] >= "2023-08-01") & 
                                 (meldungen_quartier_join["requested_datetime"] < "2023-09-01")].copy
januar = meldungen_quartier_join[(meldungen_quartier_join["requested_datetime"] >= "2023-01-01") & 
                                 (meldungen_quartier_join["requested_datetime"] < "2023-02-01")].copy


#August und Januar (create function)
#def filter_by_month(data, year = None, month = None):
#    result = []

#    for row in data:
#        if year is not None and row["requested_datetime"].year != year:
#            continue
#        if month is not None and row["requested_datetime"].month != month:
#            continue 
#        result.append(row)
        
#    return pd.DataFrame(result)

#apply function to filter data from august and january
#august_daten = filter_by_month(meldungen_quartier_join, month = 8)
#januar_daten = filter_by_month(meldungen_quartier_join, month = 1)


,objectid,requested_datetime,e,n,service_code,geometry,anzahl_meldungen,index_right,qname,qnr,kname,knr,geometry_y,anzahl_meldungen_quartier,anzahl_meldungen_kreis
4,5,2013-03-15 10:36:53,2683094,1247762,Abfall/Sammelstelle,POINT (2683094 1247762),27339,15,City,14,Kreis 1,1,"POLYGON ((2682373.2 1247057,2682399.5 1247083....",1741,5738
5,6,2013-03-16 17:54:42,2683475,1247422,Strasse/Trottoir/Platz,POINT (2683475 1247422),9870,13,Rathaus,11,Kreis 1,1,"POLYGON ((2683316.2 1247632.6,2683319.8 124763...",1457,5738
6,7,2013-03-16 18:04:21,2683303,1247675,Strasse/Trottoir/Platz,POINT (2683303 1247675),9870,24,Lindenhof,13,Kreis 1,1,"POLYGON ((2683037 1247571.6,2683044.5 1247600,...",1125,5738
14,15,2013-03-21 09:06:05,2684556,1245435,Strasse/Trottoir/Platz,POINT (2684556 1245435),9870,9,Mühlebach,82,Kreis 8,8,"POLYGON ((2683784.2 1246610.5,2683801.5 124663...",908,2997
19,20,2013-03-27 10:34:48,2683538,1247650,Strasse/Trottoir/Platz,POINT (2683538 1247650),9870,13,Rathaus,11,Kreis 1,1,"POLYGON ((2683316.2 1247632.6,2683319.8 124763...",1457,5738
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72612,72613,2026-05-01 07:48:09,2682625,1247058,Signalisation/Lichtsignal,POINT (2682625 1247058),10975,15,City,14,Kreis 1,1,"POLYGON ((2682373.2 1247057,2682399.5 1247083....",1741,5738
72613,72614,2026-05-01 08:39:59,2683133,1247858,Brunnen/Hydranten,POINT (2683133 1247858),1289,24,Lindenhof,13,Kreis 1,1,"POLYGON ((2683037 1247571.6,2683044.5 1247600,...",1125,5738
72615,72616,2026-05-01 16:23:20,2682876,1244477,Grünflächen/Spielplätze,POINT (2682876 1244477),7238,8,Wollishofen,21,Kreis 2,2,"POLYGON ((2681417 1244793.5,2681418 1244817.4,...",2837,6225
72619,72620,2026-05-01 23:33:55,2682506,1244158,Allgemein,POINT (2682506 1244158),3969,8,Wollishofen,21,Kreis 2,2,"POLYGON ((2681417 1244793.5,2681418 1244817.4,...",2837,6225


visualization to question 3

Question 4: Which trend does the august (summer) data around the lake show within the report period?

PROBLEM: CHANI ERST RICHTIG FERTIG MACHE WENN D FUNKTION VO QUESTION 3 FUNKTIONIERT

In [ ]:
import numpy as np
import pandas as pd

#check Index to ensure data is ready to resample 
print(f"August Index: {august_daten.index}") #DatetimeIndex (dtype = 'datetype64', name = 'requested_datetime')


#resample monthly mean temperatures 
august_daten["august_mean"] = august_daten["anzahl_meldungen_kreis"].resample("ME").mean().dropna()

augusts_passed = np.arange(len(august_mean))

slope, intercept = np.polyfit(augusts_passed, august_mean, 1)

print(f"long-term request trend: {slope:.3f} per year")
print(f"total change of number of requests over the dataset: {(slope * len(augusts_passed)):.2f}")

August Index: DatetimeIndex(['2023-08-08 16:39:04', '2023-08-08 22:57:52',
               '2023-08-02 16:28:36', '2023-08-04 14:45:19',
               '2023-08-09 12:48:30', '2023-08-09 12:49:06',
               '2023-08-01 09:12:51', '2023-08-01 13:49:49',
               '2023-08-01 15:31:45', '2023-08-01 15:31:59',
               ...
               '2023-08-31 16:59:47', '2023-08-31 17:00:53',
               '2023-08-31 17:01:51', '2023-08-31 17:03:08',
               '2023-08-31 17:04:41', '2023-08-31 17:07:14',
               '2023-08-31 17:08:31', '2023-08-31 18:46:36',
               '2023-08-31 20:15:23', '2023-08-31 21:31:14'],
              dtype='datetime64[us]', name='requested_datetime', length=775, freq=None)


c:\Users\User\miniconda3\envs\sds210\Lib\site-packages\numpy\lib\_polynomial_impl.py:674: RuntimeWarning: invalid value encountered in divide
  lhs /= scale


LinAlgError: SVD did not converge in Linear Least Squares